# Nuclear-to-Cell Expansion with `celldega.nbhd`

This notebook is a runnable companion to a nucleus/cell segmentation-sensitivity
analysis: starting from a nucleus polygon, grow it outward in fixed steps until it
reaches the boundary of its corresponding (larger) cell segmentation, and compute a
cell-by-gene matrix at every step -- the original nucleus radius plus each expanded
radius.

That workflow is now a first-class part of Celldega's neighborhood API:

- **`NeighborhoodCollection.calc_expansion`** replaces the manual
  `expand_nuclei_within_cell` buffering loop. It is deliberately generic: give it a
  `NeighborhoodCollection` of *any* entity and a matching per-entity bounding
  `GeoDataFrame`; it buffers every entity outward at each requested radius (in
  microns), clips each one to its own bound so growth never overshoots it, and
  returns one new `NeighborhoodCollection` per radius, all sharing the same
  observation axis so results stay directly comparable across radii. Nucleus ->
  cell is just the running example below -- the same method works for any other
  pair of nested per-entity geometries.
- **`NeighborhoodCollection.calc_signature(by="cell-free", data_dir=...)`**
  replaces the custom `assign_trx_to_entity_streaming_parquet_optimized` +
  manual pivot. It always streams a `transcripts.parquet` directory in batches
  (narrowing candidate entities per batch with a spatial index before testing
  exact polygons), so a whole-tile file doesn't need to be loaded into memory
  once per radius; `feature_col`/`x_col`/`y_col` name its gene/x/y columns
  (Xenium convention by default, but overridable for any column layout).

Because the real instrument files (OME-TIFF, per-dataset contour CSVs, a full-tile
`transcripts.parquet`) aren't available here, this notebook builds a small
**synthetic** nucleus/cell/transcript dataset with the same shape as a real
segmentation export, so every cell below runs standalone. The final section maps
each synthetic variable back to the real pipeline's inputs so you can swap in your
own paths.

In [1]:
import os
import tempfile

import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
from shapely.geometry import Point

import celldega as dega

dega.__version__

'0.18.0'

## 1. Nucleus + cell-boundary polygons

A real pipeline builds these from segmentation contour CSVs (one polygon per cell,
in each of a nucleus file and an expanded-cell-boundary file). Here we synthesize
the same shape: two `GeoDataFrame`s sharing a `cell_id` column, one with a small
nucleus polygon per cell and one with its larger enclosing cell polygon.

In [2]:
rng = np.random.default_rng(0)

N_ROWS, N_COLS = 10, 12
SPACING_UM = 20.0

records_nuclei, records_cells, cell_meta = [], [], []

cell_id = 0
for row in range(N_ROWS):
    for col in range(N_COLS):
        cx = col * SPACING_UM + rng.normal(0, 1.5)
        cy = row * SPACING_UM + rng.normal(0, 1.5)

        cell_radius = rng.uniform(7.0, 9.0)
        nucleus_radius = rng.uniform(2.5, 3.5)
        jitter = rng.uniform(0, 2.0, size=2)
        nx, ny = cx + jitter[0], cy + jitter[1]

        # two synthetic "cell types" so downstream clustering has real structure
        cell_type = "TypeA" if (row + col) % 2 == 0 else "TypeB"

        records_nuclei.append(
            {"cell_id": cell_id, "geometry": Point(nx, ny).buffer(nucleus_radius, resolution=12)}
        )
        records_cells.append(
            {"cell_id": cell_id, "geometry": Point(cx, cy).buffer(cell_radius, resolution=12)}
        )
        cell_meta.append(
            {"cell_id": cell_id, "cell_type": cell_type, "cx": cx, "cy": cy,
             "nx": nx, "ny": ny, "nucleus_radius": nucleus_radius, "cell_radius": cell_radius}
        )
        cell_id += 1

gdf_nuclei = gpd.GeoDataFrame(records_nuclei)
gdf_cells = gpd.GeoDataFrame(records_cells)
df_cell_meta = pd.DataFrame(cell_meta).set_index("cell_id")

gdf_nuclei.shape, gdf_cells.shape

((120, 2), (120, 2))

## 2. Wrap the nuclei as a `NeighborhoodCollection`

Each nucleus becomes one observation ("neighborhood"), keyed by `cell_id`.

In [3]:
nbhd_nuclei = dega.nbhd.NeighborhoodCollection(
    gdf=gdf_nuclei, nbhd_type="nucleus", nbhd_col="cell_id"
)
nbhd_nuclei.obs[["area_um2"]].head()

,area_um2
neighborhood_id,
0,19.838659
1,36.964149
2,22.426905
3,21.573983
4,37.955601


## 3. Expansion series

`calc_expansion` buffers every nucleus outward at each radius in
`radii_um` and intersects it with the matching row of `gdf_cells`, so a nucleus
never grows past its own cell's membrane. It returns a dict keyed by radius, each
value a new `NeighborhoodCollection` sharing the same `cell_id` observation axis.

In [4]:
radii_um = [0, 0.5, 1, 1.5, 2, 2.5, 3]
nbhd_series = nbhd_nuclei.calc_expansion(gdf_cells, radii_um=radii_um)

for radius, nbhd in nbhd_series.items():
    print(f"radius={radius:>4} um -> n={len(nbhd.gdf):>4}  mean_area={nbhd.gdf['area_um2'].mean():6.2f} um^2")

radius= 0.0 um -> n= 120  mean_area= 28.34 um^2
radius= 0.5 um -> n= 120  mean_area= 38.52 um^2
radius= 1.0 um -> n= 120  mean_area= 50.28 um^2
radius= 1.5 um -> n= 120  mean_area= 63.58 um^2
radius= 2.0 um -> n= 120  mean_area= 78.40 um^2
radius= 2.5 um -> n= 120  mean_area= 94.52 um^2
radius= 3.0 um -> n= 120  mean_area=111.39 um^2


In [5]:
# visual sanity check for one example cell, mirroring the original notebook's plot
example_id = str(int(df_cell_meta.index[7]))

fig, axes = plt.subplots(1, len(radii_um), figsize=(3 * len(radii_um), 3))
for ax, radius in zip(axes, radii_um):
    nbhd = nbhd_series[radius]
    gdf_cells[gdf_cells["cell_id"].astype(str) == example_id].boundary.plot(ax=ax, color="black")
    nbhd.gdf.loc[[example_id]].plot(ax=ax, color="lightblue", edgecolor="blue", alpha=0.7)
    ax.set_title(f"+{radius} um")
    ax.set_aspect("equal")
    ax.axis("off")
fig.suptitle(f"cell_id {example_id}")
fig.tight_layout()

## 4. Synthetic transcripts

Stand-in for a `transcripts.parquet` with non-Xenium columns -- `x`, `y`, `name` --
matching the columns used in the original notebook's
`assign_trx_to_entity_streaming_parquet_optimized(..., x_col="x", y_col="y",
gene_col="name")` call. Two gene pairs simulate real biology: `NucGene*`
transcripts cluster tightly at the nucleus center (captured at every radius), while
`CytoGene*` and a cell-type marker gene (`MarkerA`/`MarkerB`) scatter through the
cytoplasm and are only picked up as the buffer radius grows.

In [6]:
trx_rows = []
for cid, meta in df_cell_meta.iterrows():
    n_nuc = rng.poisson(15)
    nuc_xy = rng.normal([meta["nx"], meta["ny"]], meta["nucleus_radius"] / 3, size=(n_nuc, 2))
    nuc_genes = rng.choice(["NucGene1", "NucGene2"], size=n_nuc)

    # rejection-sample points in the cytoplasm annulus (inside cell, outside nucleus)
    cyto_xy = []
    while len(cyto_xy) < 20:
        theta = rng.uniform(0, 2 * np.pi)
        r = meta["cell_radius"] * np.sqrt(rng.uniform(0, 1))
        x, y = meta["cx"] + r * np.cos(theta), meta["cy"] + r * np.sin(theta)
        if (x - meta["nx"]) ** 2 + (y - meta["ny"]) ** 2 > meta["nucleus_radius"] ** 2:
            cyto_xy.append((x, y))
    cyto_xy = np.array(cyto_xy)
    cyto_genes = rng.choice(["CytoGene1", "CytoGene2"], size=len(cyto_xy))

    marker_gene = "MarkerA" if meta["cell_type"] == "TypeA" else "MarkerB"
    marker_xy = cyto_xy[rng.integers(0, len(cyto_xy), size=rng.poisson(10))]

    for xy, gene in zip(nuc_xy, nuc_genes):
        trx_rows.append({"x": xy[0], "y": xy[1], "name": gene})
    for xy, gene in zip(cyto_xy, cyto_genes):
        trx_rows.append({"x": xy[0], "y": xy[1], "name": gene})
    for xy in marker_xy:
        trx_rows.append({"x": xy[0], "y": xy[1], "name": marker_gene})

df_trx = pd.DataFrame(trx_rows)

# calc_signature's cell-free mode always streams from a transcripts.parquet on
# disk, so persist these to a directory rather than keeping them in memory
trx_dir = tempfile.mkdtemp()
df_trx.to_parquet(f"{trx_dir}/transcripts.parquet")
df_trx.shape, df_trx["name"].value_counts().to_dict()

((5490, 3),
 {np.str_('CytoGene1'): 1243,
  np.str_('CytoGene2'): 1157,
  np.str_('NucGene1'): 984,
  np.str_('NucGene2'): 837,
  'MarkerA': 644,
  'MarkerB': 625})

## 5. Cell-by-gene matrix at every radius

`calc_signature(by="cell-free", data_dir=...)` spatially joins transcripts to
each radius's polygons and returns transcript counts as an `AnnData` in
`nbhd.mod["gene_cell_free"]` -- one call per radius, no custom pivot code
needed. `data_dir` points to a directory containing a `transcripts.parquet`;
`feature_col`/`x_col`/`y_col` name its gene/x/y columns (Xenium convention by
default, overridden below for this notebook's custom `name`/`x`/`y` columns).
The file is always streamed in batches internally -- narrowing candidate
entities per batch with a spatial index before testing exact polygons -- so a
whole-tile file doesn't need to be loaded into memory once per radius.

In [7]:
for radius, nbhd in nbhd_series.items():
    nbhd.calc_signature(
        by="cell-free", data_dir=trx_dir, feature_col="name", x_col="x", y_col="y",
        drop_missing=False,
    )

gene_totals = pd.DataFrame(
    {
        radius: pd.DataFrame(
            nbhd.mod["gene_cell_free"].X, columns=nbhd.mod["gene_cell_free"].var_names
        ).sum()
        for radius, nbhd in nbhd_series.items()
    }
).T
gene_totals

Calculating neighborhood-by-gene (cell-free, streaming)
Calculating neighborhood-by-gene (cell-free, streaming)
Calculating neighborhood-by-gene (cell-free, streaming)
Calculating neighborhood-by-gene (cell-free, streaming)
Calculating neighborhood-by-gene (cell-free, streaming)
Calculating neighborhood-by-gene (cell-free, streaming)
Calculating neighborhood-by-gene (cell-free, streaming)


,CytoGene1,CytoGene2,MarkerA,MarkerB,NucGene1,NucGene2
0.0,NaN,NaN,NaN,NaN,970.0,824.0
0.5,82.0,79.0,37.0,41.0,982.0,834.0
1.0,171.0,162.0,76.0,75.0,984.0,837.0
1.5,274.0,255.0,118.0,121.0,984.0,837.0
2.0,379.0,367.0,170.0,172.0,984.0,837.0
2.5,485.0,463.0,228.0,217.0,984.0,837.0
3.0,610.0,571.0,291.0,284.0,984.0,837.0


## Mapping this onto a real pipeline

| Original notebook | This notebook / Celldega API |
| --- | --- |
| `gdf_nuclei_original` (parsed from `..._nuclei_contour_coords.csv`) | `gdf_nuclei` -> `NeighborhoodCollection(gdf=gdf_nuclei, nbhd_col="cell_id")` |
| `gdf_cells` / `gdf_cells2` (parsed from `..._Expanded_5um_cell_contour_coords.csv`) | `gdf_cells` passed to `calc_expansion` |
| `expand_nuclei_within_cell(nuclei_gdf, expand_um)` loop building `nuclei_gdfs = {"original": ..., "expanded_0_5um": ..., ...}` | `nbhd_series = nbhd_nuclei.calc_expansion(gdf_cells, radii_um=[0, 0.5, 1, 1.5, 2, 2.5, 3])` |
| `assign_trx_to_entity_streaming_parquet_optimized(trx_parquet_path, entity_gdf, x_col="x", y_col="y", gene_col="name", batch_size=1_000_000)` + manual `pivot_table` per radius | `nbhd.calc_signature(by="cell-free", data_dir=trx_dir, feature_col="name", x_col="x", y_col="y")` per radius -- same batched-parquet-plus-spatial-index mechanics, now built in and always used |
| Per-radius `pd.read_parquet(..._nuclei_by_gene.parquet)` -> `AnnData` (e.g. `ad.AnnData(X=nbg)`) | `nbhd.mod["gene_cell_free"]` (already an `AnnData`) |
| Per-radius `adata.write(...h5ad)` | `nbhd.mod["gene_cell_free"].write_h5ad(...)`, or persist the whole collection (geometry + all modalities) with `nbhd.write("radius.h5mu")` |
| `safe_polygon`, `simple_format`, `transform_polygon`, `make_column_names_unique_fast` helper functions | available as `celldega.nbhd.safe_polygon` / `simple_format` / `transform_polygon` / `make_column_names_unique_fast`, unchanged -- not otherwise used in this notebook |

`calc_signature`'s `by="cell-free"` mode always streams from a
`transcripts.parquet` under `data_dir=` in batches (narrowing candidate
entities per batch with a spatial index before testing exact polygons), so a
whole-tile file re-joined once per radius across an expansion series doesn't
need to fit in memory. `feature_col`/`x_col`/`y_col` name its columns --
Xenium convention (`feature_name`/`x_location`/`y_location`) by default, or
whatever your own `transcripts.parquet` uses.